# Differential Gene Expression Analysis: RNA-seq with PyDESeq2
**Dataset:** Airway smooth muscle cell RNA-seq (dexamethasone treatment vs. control)  
**Pipeline:** PyDESeq2 · Gene annotation · PCA · Gene Set Enrichment Analysis (GSEA) · Volcano plot · Heatmap

---

## Overview

This notebook implements a complete RNA-seq differential gene expression pipeline in Python, replicating a workflow traditionally done in R using DESeq2. We use **PyDESeq2**, a Python reimplementation of the DESeq2 statistical model, to identify genes that are significantly differentially expressed between dexamethasone-treated and untreated airway smooth muscle cells.

**Pipeline steps:**
1. Load raw RNA-seq count data and sample metadata
2. Run differential expression analysis with PyDESeq2
3. Annotate results with human-readable gene symbols
4. Visualize sample clustering with PCA
5. Run Gene Set Enrichment Analysis (GSEA) against KEGG pathways
6. Visualize results with a volcano plot and expression heatmap

**Dataset background:** the airway dataset is a well-known RNA-seq benchmark comparing airway smooth muscle cells treated with dexamethasone (a corticosteroid used in asthma treatment) against untreated controls. Source: [BRITE-REU programming workshops](https://github.com/BRITE-REU/programming-workshops).

## 1. Setup and Data Loading

In [ ]:
# Download the raw count matrix and sample metadata
!wget -q 'https://raw.githubusercontent.com/BRITE-REU/programming-workshops/master/source/workshops/02_R/files/airway_scaledcounts.csv'
!wget -q 'https://raw.githubusercontent.com/BRITE-REU/programming-workshops/master/source/workshops/02_R/files/airway_metadata.csv'

In [ ]:
# Install required packages
%pip install pydeseq2 -q
%pip install scanpy -q
%pip install gseapy -q
%pip install sanbomics -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

from sanbomics.tools import id_map
from sanbomics.plots import volcano

import gseapy as gp
from gseapy import dotplot

### 1.1 Load the count matrix and metadata

The count matrix contains raw read counts: one row per gene, one column per sample. The metadata table describes each sample's experimental condition (treated with dexamethasone, or control).

**Note:** PyDESeq2 documentation: https://pydeseq2.readthedocs.io/en/latest/

In [ ]:
counts = pd.read_csv('airway_scaledcounts.csv', index_col=0)
print(f"Count matrix shape: {counts.shape}  (genes x samples)")
counts.head()

In [ ]:
# Remove genes with zero counts across all samples — these carry no
# information and would only add noise/computation time to the model
counts = counts[counts.sum(axis=1, numeric_only=True) > 0]
print(f"Genes remaining after filtering zero-count genes: {counts.shape[0]:,}")

In [ ]:
metadata = pd.read_csv('airway_metadata.csv', index_col=0)
print(f"Metadata shape: {metadata.shape}")
metadata

In [ ]:
# Simplify metadata to just the columns we need: sample ID (index) and treatment (dex)
metadata_clean = metadata[['dex']].copy()
print(f"Samples: {metadata_clean.shape[0]}")
print(f"\nTreatment groups:\n{metadata_clean['dex'].value_counts()}")
metadata_clean

**Important:** PyDESeq2 expects the count matrix oriented as **samples × genes**, the opposite of how it's typically stored (genes × samples). We transpose when building the `DeseqDataSet` below.

## 2. Differential Expression Analysis with PyDESeq2

PyDESeq2 models raw RNA-seq counts using a negative binomial distribution and tests for differential expression between conditions, accounting for differences in sequencing depth (library size) and gene-wise dispersion. This is the same statistical model used by the widely-cited R package DESeq2.

**`design_factors`** specifies which metadata column defines our experimental groups — here, `dex` (dexamethasone treatment status).

In [ ]:
dds = DeseqDataSet(
    counts=counts.T,            # transpose to samples x genes, as PyDESeq2 expects
    metadata=metadata_clean,
    design_factors="dex"
)

# Run the full DESeq2 pipeline: size factor estimation, dispersion estimation,
# and negative binomial GLM fitting
dds.deseq2()

### 2.1 Extract statistical results

`DeseqStats` computes the actual differential expression test. The **contrast** argument specifies the comparison direction — here, `control` is the baseline and `treated` is the condition being compared against it. Getting the contrast order backwards doesn't break anything, but it does flip the sign of the fold change, so it's worth double-checking.

In [ ]:
stat_res = DeseqStats(dds, contrast=('dex', 'control', 'treated'))
stat_res.summary()

In [ ]:
res = stat_res.results_df
print(f"Total genes tested: {len(res):,}")
res.head()

## 3. Gene Annotation

Raw RNA-seq results are indexed by Ensembl gene IDs (e.g., `ENSG00000000003`), which aren't human-readable. We map these to standard gene symbols (e.g., `TSPAN6`) using `sanbomics`, which wraps common annotation databases.

In [ ]:
mapper = id_map(species='human')
res['Symbol'] = res.index.map(mapper.mapper)

print("Results with gene symbols added:")
res.head()

### 3.1 Filter for significant genes

We apply standard significance thresholds: adjusted p-value (padj) < 0.05 and an absolute log2 fold change > 0.5 (roughly a 41% change in expression in either direction).

In [ ]:
PADJ_THRESHOLD = 0.05
FC_THRESHOLD   = 0.5

sigs = res[(res.padj < PADJ_THRESHOLD) & (abs(res.log2FoldChange) > FC_THRESHOLD)]

print(f"Significant genes: {len(sigs):,} out of {len(res):,} tested")
print(f"  Upregulated (treated > control):   {(sigs.log2FoldChange > 0).sum():,}")
print(f"  Downregulated (treated < control): {(sigs.log2FoldChange < 0).sum():,}")

sigs.sort_values('padj').head(10)

## 4. Principal Component Analysis — Sample Clustering

Before trusting differential expression results, it's good practice to check whether samples cluster by treatment group, as expected. PCA on the normalized expression data reduces thousands of genes down to a 2D projection, making sample relationships easy to visualize.

We use **scanpy** (`sc.tl.pca` and `sc.pl.pca`) since the `DeseqDataSet` object is structured similarly to an AnnData object.

In [ ]:
sc.tl.pca(dds)
sc.pl.pca(dds, color='dex', size=200)

If treated and control samples form visually distinct clusters, that's a good sign — it suggests the dexamethasone treatment has a genuine, detectable effect on the overall transcriptome, supporting confidence in the differential expression results above.

## 5. Gene Set Enrichment Analysis (GSEA)

Looking at individual differentially expressed genes only tells part of the story. GSEA asks a complementary question: are genes belonging to known **biological pathways** systematically shifted toward up- or down-regulation, even if no single gene in that pathway is individually significant?

We rank all genes by their DESeq2 test statistic (`stat`), then test whether genes from known KEGG pathways are enriched toward either end of that ranking.

Documentation: https://gseapy.readthedocs.io/en/latest/introduction.html

In [ ]:
# Build a ranked gene list using the DESeq2 test statistic
# Genes with the strongest evidence of upregulation rank highest;
# strongest downregulation ranks lowest
ranking = (
    res[['Symbol', 'stat']]
    .dropna()
    .drop_duplicates('Symbol')
    .sort_values('stat', ascending=False)
    .reset_index()[['Symbol', 'stat']]
)

print(f"Genes in ranked list: {len(ranking):,}")
ranking.head(10)

In [ ]:
# Run GSEA pre-ranked analysis against the KEGG 2021 Human pathway database
pre_res = gp.prerank(
    rnk=ranking,
    gene_sets=['KEGG_2021_Human'],
    seed=6,
    permutation_num=100
)

In [ ]:
# Compile enrichment results into a clean summary table
enrichment_results = []
for term in list(pre_res.results):
    enrichment_results.append([
        term,
        pre_res.results[term]['fdr'],
        pre_res.results[term]['es'],
        pre_res.results[term]['nes']
    ])

enrichment_df = pd.DataFrame(
    enrichment_results,
    columns=['Term', 'fdr', 'es', 'nes']
).sort_values('fdr').reset_index(drop=True)

print("Top enriched pathways (by FDR):")
enrichment_df.head(10)

**Interpreting the columns:**
- **es (enrichment score):** measures how strongly genes from this pathway cluster at the top or bottom of the ranked list
- **nes (normalized enrichment score):** the ES adjusted for gene set size, allowing comparison across pathways of different sizes
- **fdr:** false discovery rate-corrected significance — lower values indicate stronger evidence of real enrichment

### 5.1 Visualize the top enriched pathway

In [ ]:
top_term = enrichment_df['Term'][0]
print(f"Top enriched pathway: {top_term}")

terms = pre_res.res2d.Term
axs = pre_res.plot(
    terms=terms[0],
    show_ranking=True,
    figsize=(5, 6)
)

In [ ]:
# Visualize the top 10 enriched pathways together
axs = pre_res.plot(
    terms=terms[0:10],
    show_ranking=True,
    figsize=(5, 6)
)

### 5.2 Dot plot summary of enriched pathways

In [ ]:
ax = dotplot(
    pre_res.res2d,
    column="FDR q-val",
    title='KEGG 2021 Human — Pathway Enrichment',
    size=6,
    figsize=(4, 5),
    cutoff=0.25,
    show_ring=False
)

## 6. Volcano Plot

A volcano plot summarizes the entire differential expression result in one figure: log2 fold change on the x-axis, statistical significance on the y-axis. Genes in the upper-left and upper-right corners are both highly significant and show a large magnitude of change — the strongest candidates for biological follow-up.

In [ ]:
volcano(res, symbol='Symbol')

## 7. Heatmap — Top Differentially Expressed Genes

A heatmap of the top differentially expressed genes across all samples shows both the magnitude of expression change and how consistent that pattern is across replicates within each treatment group.

In [ ]:
# Take the top 50 most significant genes by adjusted p-value
top_genes = sigs.sort_values('padj').head(50)

# Pull normalized counts for these genes from the DeseqDataSet
# dds.layers['normed_counts'] contains size-factor-normalized expression values
normed_counts = pd.DataFrame(
    dds.layers['normed_counts'],
    index=dds.obs_names,
    columns=dds.var_names
).T

heatmap_data = normed_counts.loc[top_genes.index]

# Replace Ensembl IDs with gene symbols for readability where available
heatmap_data.index = top_genes['Symbol'].fillna(top_genes.index).values

# Log-transform for visualization (expression data is highly right-skewed)
heatmap_data_log = np.log2(heatmap_data + 1)

plt.figure(figsize=(8, 12))
sns.heatmap(
    heatmap_data_log,
    cmap='RdBu_r',
    center=heatmap_data_log.values.mean(),
    yticklabels=True,
    xticklabels=True,
    cbar_kws={'label': 'log2(normalized counts + 1)'}
)
plt.title('Top 50 Differentially Expressed Genes\n(Dexamethasone Treated vs. Control)')
plt.xlabel('Sample')
plt.ylabel('Gene')
plt.tight_layout()
plt.show()

## 8. Summary

This notebook implemented a complete RNA-seq differential expression pipeline in Python:

1. **Data preparation** — loaded raw counts and metadata, filtered zero-count genes
2. **Differential expression** — used PyDESeq2 (a Python implementation of the DESeq2 negative binomial model) to test for differential expression between dexamethasone-treated and control airway cells
3. **Gene annotation** — mapped Ensembl gene IDs to readable gene symbols
4. **Quality control** — verified treatment groups separate cleanly via PCA, supporting confidence in the differential expression results
5. **Pathway-level analysis** — ran GSEA against KEGG pathways to identify systematically enriched biological processes, beyond individual gene-level significance
6. **Visualization** — built a volcano plot summarizing all genes, and a heatmap of the top 50 differentially expressed genes across samples

**Key takeaways:**
- PyDESeq2 makes it possible to run a full DESeq2-style analysis without leaving Python, useful for integrating RNA-seq results into broader Python-based bioinformatics pipelines
- PCA is a valuable sanity check before trusting differential expression results — if treatment groups don't separate at all, something may be wrong upstream
- GSEA captures pathway-level signal that individual gene-level testing can miss, especially when many genes in a pathway shift modestly rather than a few shifting dramatically
- Significance thresholds (padj, fold change) should always be stated explicitly rather than left implicit in the code

**Potential extensions:**
- Compare results against the original R DESeq2 output to validate consistency between implementations
- Extend GSEA to additional gene set databases (GO Biological Process, Reactome, Hallmark gene sets)
- Apply batch correction if samples come from multiple sequencing runs or experimental batches